<a href="https://colab.research.google.com/github/saisai257274/NLP-1/blob/main/nlp_ass_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Reasoning**:
I will import the pandas library and create a DataFrame with the specified news headlines to simulate an Excel-like structure as requested in the instructions.



In [7]:
import pandas as pd

# Create a list of new news headlines
new_headlines = [
    'Virat scored century in match',
    'BJP won in elections',
    'Tech giant announces new smartphone',
    'Stock market hits record high',
    'SpaceX successfully lands rocket'
]

# Convert the list into a pandas DataFrame with a single column 'Headline'
df_new_headlines = pd.DataFrame(new_headlines, columns=['Headline'])

# Display the DataFrame to verify its content and structure
print("New Headlines DataFrame:")
df_new_headlines

New Headlines DataFrame:


,Headline
0,Virat scored century in match
1,BJP won in elections
2,Tech giant announces new smartphone
3,Stock market hits record high
4,SpaceX successfully lands rocket


## Text Preprocessing

### Subtask:
Clean the new news headlines, handle word normalization, and prepare the text for vectorization.


**Reasoning**:
I will define the text preprocessing function and apply it to the new headlines DataFrame to clean the text, remove non-alphabetic characters, and normalize whitespace as per the instructions.



In [8]:
import re

def preprocess_text(text):
    # Convert to lowercase and remove all characters that are not lowercase letters or spaces
    cleaned = re.sub(r'[^a-z\s]', '', text.lower())
    # Normalize whitespace using split and join
    return ' '.join(cleaned.split())

# Create the 'Cleaned_Headline' column by applying the preprocessing function
df_new_headlines['Cleaned_Headline'] = df_new_headlines['Headline'].apply(preprocess_text)

# Print the first few rows to verify the transformation
print('Updated New Headlines DataFrame:')
df_new_headlines.head()

Updated New Headlines DataFrame:


,Headline,Cleaned_Headline
0,Virat scored century in match,virat scored century in match
1,BJP won in elections,bjp won in elections
2,Tech giant announces new smartphone,tech giant announces new smartphone
3,Stock market hits record high,stock market hits record high
4,SpaceX successfully lands rocket,spacex successfully lands rocket


## Generate BoW DataFrame

### Subtask:
Create a Bag of Words (BoW) representation of the preprocessed headlines and convert it into a structured pandas DataFrame.


**Reasoning**:
I will import the CountVectorizer from scikit-learn, transform the cleaned headlines into a Bag of Words representation, and then create a new DataFrame to store and display these word counts.



In [9]:
from sklearn.feature_extraction.text import CountVectorizer

# Instantiate CountVectorizer with English stop words
vectorizer = CountVectorizer(stop_words='english')

# Fit and transform the preprocessed headlines
bow_matrix = vectorizer.fit_transform(df_new_headlines['Cleaned_Headline'])

# Retrieve feature names (words)
feature_names = vectorizer.get_feature_names_out()

# Create a DataFrame representing the Bag of Words
df_new_bow = pd.DataFrame(bow_matrix.toarray(), columns=feature_names)

# Display the BoW DataFrame
print('Bag of Words (BoW) DataFrame:')
df_new_bow

Bag of Words (BoW) DataFrame:


,announces,bjp,century,elections,giant,high,hits,lands,market,match,...,record,rocket,scored,smartphone,spacex,stock,successfully,tech,virat,won
0,0,0,1,0,0,0,0,0,0,1,...,0,0,1,0,0,0,0,0,1,0
1,0,1,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,1,0,0,0,1,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0
3,0,0,0,0,0,1,1,0,1,0,...,1,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,0,0,1,0,0,...,0,1,0,0,1,0,1,0,0,0


## Apply LDA Modeling

### Subtask:
Initialize and fit a Latent Dirichlet Allocation (LDA) model to discover the latent topics in the new dataset.


**Reasoning**:
I will import the LatentDirichletAllocation class, instantiate the model with 2 topics, fit it to the BoW matrix, and then transform the matrix to obtain the topic distributions for each headline.



In [10]:
from sklearn.decomposition import LatentDirichletAllocation

# Instantiate the LDA model with 2 topics
lda_model = LatentDirichletAllocation(n_components=2, random_state=42)

# Fit the model using the previously generated bow_matrix
lda_model.fit(bow_matrix)

# Transform the bow_matrix to get topic distributions for each headline
lda_output = lda_model.transform(bow_matrix)

# Print the shape of the output to verify
print('LDA Output Shape:', lda_output.shape)
print('LDA Output (Topic Distributions):')
print(lda_output)

LDA Output Shape: (5, 2)
LDA Output (Topic Distributions):
[[0.10296846 0.89703154]
 [0.12873124 0.87126876]
 [0.91130578 0.08869422]
 [0.91130578 0.08869422]
 [0.89350385 0.10649615]]


## Identify Topic Keywords

### Subtask:
Extract and display the top words associated with each discovered topic to understand the themes.


**Reasoning**:
I will extract the top 5 keywords for each topic by accessing the `components_` of the LDA model, sorting them to find the highest weights, and mapping those indices back to the feature names.



In [11]:
import numpy as np

# Function to display the top keywords for each topic
def display_topics(model, feature_names, no_top_words):
    for topic_idx, topic in enumerate(model.components_):
        # Get indices of the top words by weight and reverse the order
        top_indices = topic.argsort()[::-1][:no_top_words]
        # Map indices to words
        top_words = [feature_names[i] for i in top_indices]
        print(f'Topic {topic_idx}: {", ".join(top_words)}')

# Display top 5 words for each topic
print('Top 5 Keywords for Each Topic:')
display_topics(lda_model, feature_names, 5)

Top 5 Keywords for Each Topic:
Topic 0: record, high, hits, stock, market
Topic 1: virat, match, century, scored, elections


## Assign Topics to Headlines

### Subtask:
Map the dominant topic back to each original headline in the DataFrame.


**Reasoning**:
I will determine the dominant topic for each headline by finding the index of the maximum probability in the LDA output and then add this information as a new column to the headlines DataFrame.



In [12]:
import numpy as np

# Determine the dominant topic for each headline using argmax on the LDA output
dominant_topics = np.argmax(lda_output, axis=1)

# Add a new column 'Topic' to the df_new_headlines DataFrame
df_new_headlines['Topic'] = dominant_topics

# Display the updated DataFrame to verify the topic assignment
print('Updated Headlines with Assigned Topics:')
df_new_headlines

Updated Headlines with Assigned Topics:


,Headline,Cleaned_Headline,Topic
0,Virat scored century in match,virat scored century in match,1
1,BJP won in elections,bjp won in elections,1
2,Tech giant announces new smartphone,tech giant announces new smartphone,0
3,Stock market hits record high,stock market hits record high,0
4,SpaceX successfully lands rocket,spacex successfully lands rocket,0


## Summary:

### Data Analysis Key Findings

* **Dataset and Vocabulary**: The analysis was performed on 5 news headlines. After removing English stop words and non-alphabetic characters, the corpus was reduced to a vocabulary of 21 unique keywords.
* **Topic Discovery**: The Latent Dirichlet Allocation (LDA) model successfully identified 2 latent topics within the small dataset:
    * **Topic 0 (Finance/Technology/Space)**: Characterized by keywords such as *record*, *high*, *hits*, *stock*, and *market*.
    * **Topic 1 (Sports/Politics)**: Characterized by keywords such as *virat*, *match*, *century*, *scored*, and *elections*.
* **Classification Results**:
    * Three headlines ("Tech giant announces new smartphone", "Stock market hits record high", and "SpaceX successfully lands rocket") were mapped to **Topic 0**.
    * Two headlines ("Virat scored century in match" and "BJP won in elections") were mapped to **Topic 1**.
* **Model Specifics**: The LDA model output a probability distribution for each headline across the 2 topics, with most headlines showing a strong dominant association with one specific topic (e.g., Topic 1 for the first two entries).

### Insights or Next Steps

* **Dataset Expansion**: The current model grouped sports and politics together into Topic 1. To achieve higher granularity and separate these distinct categories, a larger dataset and an increased number of components (`n_components`) in the LDA model are recommended.
* **Refining Preprocessing**: While basic cleaning was effective, incorporating Lemmatization or Stemming in future iterations could further consolidate related words (e.g., "lands" and "landing") to improve topic coherence.
